# Inference
Load all models and run inference over the evaluation set. Results are saved to `data/eval/inference_results.pkl`.

Re-run this notebook when models or audio data change. The evaluation notebook (`full_evaluation.ipynb`) loads the saved pickle and does not need the models.

# 1 Imports & Setup

In [1]:
import sys, pathlib

PROJECT_ROOT = pathlib.Path('.').resolve()
while not (PROJECT_ROOT / '.git').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
import torchaudio
import soundfile as sf
from transformers import AutoProcessor, AutoModelForCTC

In [3]:
import json

from scripts.eval.eval import (
    evaluate_mispronunciation_detection,
    show_alignment,
    show_alignment_html,
    forced_ctc_phones,
)

from scripts.data_handling.collapse_phonemes import collapse_phones
from scripts.asr_system.ensemble.ensemble_inference import run_ensemble_inference
from scripts.asr_system.ensemble.ensemble import (
    build_ru_ipa_dict,
    build_pal_set,
    frame_gibbs_confidence,
)

In [ ]:
DEVICE        = "cpu" # "cuda" if torch.cuda.is_available() else "cpu" decomment if running on kaggle.
SEGMENTS_DIR  = PROJECT_ROOT / "data" / "eval" / "segments"
IMAGES_DIR    = PROJECT_ROOT / "paper" / "0000-CadyThesis-Draft" / "images"

GA_MODEL_ID           = "duck-hug-567/xls-r-300m-gaphon-collapsed-v2"#"duck-hug-567/xls-r-300m-gaphon-full-collapsed"
EN_MODEL_ID           = "duck-hug-567/xls-r-300m-engphon-collapsed-v2"
RU_MODEL_ID           = "snu-nia-12/wav2vec-large-xlsr-53_nia12_phone-nsu-ai-_russian"  # set to model ID string to enable Russian specialist
L2_MODEL_ID           = "duck-hug-567/xls-r-300m-synth_paired-v4"  # synthetic-augmented model (paired training)
L2_BASELINE_MODEL_ID  = "duck-hug-567/xls-r-300m-unpaired-v2"  # synthetic-augmented model (unpaired baseline)
COMBINED_MODEL_ID     = "duck-hug-567/xls-r-300m-eng-irish-combined-v2"  # single model trained on Irish+English data combined

print(f"Device:       {DEVICE}")
print(f"Segments dir: {SEGMENTS_DIR}")
print(f"Images dir:   {IMAGES_DIR}")

Device:       cpu
Segments dir: /home/peter/Desktop/thesis/ThesisProject/data/eval/segments
Images dir:   /home/peter/Desktop/thesis/ThesisProject/paper/0000-CadyThesis-Draft/images


In [ ]:
import joblib

sel_prob         = joblib.load(PROJECT_ROOT / 'models/ensemble/lr_selector_prob.pkl')
sel_prob_pooled  = joblib.load(PROJECT_ROOT / 'models/ensemble/lr_selector_prob_pooled.pkl')
sel_gibbs        = joblib.load(PROJECT_ROOT / 'models/ensemble/lr_selector_gibbs.pkl')
sel_gibbs_pooled = joblib.load(PROJECT_ROOT / 'models/ensemble/lr_selector_gibbs_pooled.pkl')
print("Selectors loaded.")

Selectors loaded.


In [6]:
ga_processor = AutoProcessor.from_pretrained(GA_MODEL_ID)
ga_model     = AutoModelForCTC.from_pretrained(GA_MODEL_ID).to(DEVICE).eval()

en_processor = AutoProcessor.from_pretrained(EN_MODEL_ID)
en_model     = AutoModelForCTC.from_pretrained(EN_MODEL_ID).to(DEVICE).eval()

ru_processor = ru_model = None
if RU_MODEL_ID:
    ru_processor = AutoProcessor.from_pretrained(RU_MODEL_ID)
    ru_model     = AutoModelForCTC.from_pretrained(RU_MODEL_ID).to(DEVICE).eval()

l2_processor = l2_model = None
if L2_MODEL_ID:
    l2_processor = AutoProcessor.from_pretrained(L2_MODEL_ID)
    l2_model     = AutoModelForCTC.from_pretrained(L2_MODEL_ID).to(DEVICE).eval()

l2_baseline_processor = l2_baseline_model = None
if L2_BASELINE_MODEL_ID:
    l2_baseline_processor = AutoProcessor.from_pretrained(L2_BASELINE_MODEL_ID)
    l2_baseline_model     = AutoModelForCTC.from_pretrained(L2_BASELINE_MODEL_ID).to(DEVICE).eval()

combined_processor = combined_model = None
if COMBINED_MODEL_ID:
    combined_processor = AutoProcessor.from_pretrained(COMBINED_MODEL_ID)
    combined_model     = AutoModelForCTC.from_pretrained(COMBINED_MODEL_ID).to(DEVICE).eval()

print("Models loaded.")


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Models loaded.


# 2 Load Data
Annotations (FLEURS Gold standard), Ulster transcriptions (canonical), audio (FLEURS eval audio)

In [7]:
with open(SEGMENTS_DIR / "manifest.json") as f:
    records = json.load(f)

print(f"Loaded {len(records)} segments")
print(f"Total canonical phones : {sum(len(r['canonical']) for r in records)}")
print(f"Total gold phones      : {sum(len(r['gold']) for r in records)}")

Loaded 148 segments
Total canonical phones : 987
Total gold phones      : 985


In [8]:

records[1]

{'audio_id': '10090378450980723387_xlyMR',
 'audio_path': '/home/peter/Desktop/thesis/ThesisProject/data/eval/segments/audio/10090378450980723387_xlyMR.wav',
 'canonical': ['lʲ', 'oː', 'nʲ'],
 'gold': ['lʲ', 'oi', 'nʲ']}

## 2.1 Normalise

Apply `collapse_phones` to canonical and gold so all three sequences (canonical, gold, asr)
share the same phone space — the ga model's collapsed vocab.
The asr output from inference is already in this space by construction.

In [9]:
for r in records:
    r['canonical'] = collapse_phones(r['canonical'])
    r['gold']      = collapse_phones(r['gold'])

worth noting that this normalization is performed in place, so if I need access to the originals, I need ot reload the fetching.

## 2.2 Sanity check

In [10]:
for r in records:
    print(f"\n{r['audio_id']}")
    print(f"  canonical ({len(r['canonical'])}): {' '.join(r['canonical'][:10])} ...")
    print(f"  gold      ({len(r['gold'])}): {' '.join(r['gold'][:10])} ...")


10090378450980723387_HZHjy
  canonical (4): i s ia d ...
  gold      (4): ɪ s i d ...

10090378450980723387_xlyMR
  canonical (3): lʲ oː nʲ ...
  gold      (3): lʲ oi nʲ ...

10090378450980723387_BPgpc
  canonical (5): n ə k a tʲ ...
  gold      (5): n ə x a tʲ ...

10090378450980723387_I1qHG
  canonical (9): i s s oː ʃ ia l t ə ...
  gold      (8): i s o l ai nʲ tʲ ə ...

10090378450980723387_Sy9Eg
  canonical (8): m a ɾʲ ə n ʃ iː d ...
  gold      (7): m ɑ ɻ ə n iː d ...

10090378450980723387_zEKNj
  canonical (6): ŋ ɡ ɾ uː p iː ...
  gold      (6): n a ɹ uː p iː ...

10090378450980723387_fVLli
  canonical (4): m oː ɾ ə ...
  gold      (4): m oː ɹ ə ...

10090378450980723387_O_AjK
  canonical (4): a l t iː ...
  gold      (5): e a l t iː ...

10200386800286571225_L6jSP
  canonical (5): x o mʲ a d ...
  gold      (5): k ə vʲ a d ...

10200386800286571225_KhX8-
  canonical (11): p aː ɾ tʲ iː ɾ iˑə l ə h ...
  gold      (13): p aː ɾ tʲ iː n a ɾ i l ...

10200386800286571225_5MDGX
  can

# 3 Inference

## 3.1 Ensemble

### 3.1.1 No selector

#### 2-way

In [ ]:
from scripts.asr_system.ensemble.ensemble import frame_prob_confidence

# ── Gibbs confidence ──────────────────────────────────────────────────────────
ensemble_2way_records = run_ensemble_inference(
    records, ga_processor, ga_model, en_processor, en_model,
    audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
)
print(f"Ensemble (2-way, gibbs) done. {len(ensemble_2way_records)} segments.")

ensemble_2way_pooled_records = run_ensemble_inference(
    records, ga_processor, ga_model, en_processor, en_model,
    pool_ga=True,
    audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
)
print(f"Ensemble (2-way, gibbs+pooled) done. {len(ensemble_2way_pooled_records)} segments.")

# ── Probability confidence ────────────────────────────────────────────────────
ensemble_2way_prob_records = run_ensemble_inference(
    records, ga_processor, ga_model, en_processor, en_model,
    conf_func=frame_prob_confidence,
    audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
)
print(f"Ensemble (2-way, prob) done. {len(ensemble_2way_prob_records)} segments.")

ensemble_2way_prob_pooled_records = run_ensemble_inference(
    records, ga_processor, ga_model, en_processor, en_model,
    conf_func=frame_prob_confidence, pool_ga=True,
    audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
)
print(f"Ensemble (2-way, prob+pooled) done. {len(ensemble_2way_prob_pooled_records)} segments.")

#### 3-way

In [ ]:
ensemble_3way_records         = None
ensemble_3way_pooled_records  = None
ensemble_3way_prob_records    = None
ensemble_3way_prob_pooled_records = None

if ru_processor is not None:
    # ── Gibbs confidence ──────────────────────────────────────────────────────
    ensemble_3way_records = run_ensemble_inference(
        records, ga_processor, ga_model, en_processor, en_model,
        ru_processor=ru_processor, ru_model=ru_model,
        audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
    )
    print(f"Ensemble (3-way, gibbs) done. {len(ensemble_3way_records)} segments.")

    ensemble_3way_pooled_records = run_ensemble_inference(
        records, ga_processor, ga_model, en_processor, en_model,
        ru_processor=ru_processor, ru_model=ru_model,
        pool_ga=True,
        audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
    )
    print(f"Ensemble (3-way, gibbs+pooled) done. {len(ensemble_3way_pooled_records)} segments.")

    # ── Probability confidence ────────────────────────────────────────────────
    ensemble_3way_prob_records = run_ensemble_inference(
        records, ga_processor, ga_model, en_processor, en_model,
        ru_processor=ru_processor, ru_model=ru_model,
        conf_func=frame_prob_confidence,
        audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
    )
    print(f"Ensemble (3-way, prob) done. {len(ensemble_3way_prob_records)} segments.")

    ensemble_3way_prob_pooled_records = run_ensemble_inference(
        records, ga_processor, ga_model, en_processor, en_model,
        ru_processor=ru_processor, ru_model=ru_model,
        conf_func=frame_prob_confidence, pool_ga=True,
        audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
    )
    print(f"Ensemble (3-way, prob+pooled) done. {len(ensemble_3way_prob_pooled_records)} segments.")
else:
    print("RU_MODEL_ID not set — 3-way ensembles skipped.")

### 3.1.2 With selector

#### 2-way

In [ ]:
# ── Gibbs confidence ──────────────────────────────────────────────────────────
ens_sel_gibbs_records = run_ensemble_inference(
    records, ga_processor, ga_model, en_processor, en_model,
    conf_func=frame_gibbs_confidence,
    selector=sel_gibbs,
    audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
)
print(f"Ensemble (2-way, sel/gibbs) done. {len(ens_sel_gibbs_records)} segments.")

ens_sel_gibbs_pooled_records = run_ensemble_inference(
    records, ga_processor, ga_model, en_processor, en_model,
    conf_func=frame_gibbs_confidence, pool_ga=True,
    selector=sel_gibbs_pooled,
    audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
)
print(f"Ensemble (2-way, sel/gibbs+pooled) done. {len(ens_sel_gibbs_pooled_records)} segments.")

# ── Probability confidence ────────────────────────────────────────────────────
ens_sel_prob_records = run_ensemble_inference(
    records, ga_processor, ga_model, en_processor, en_model,
    conf_func=frame_prob_confidence,
    selector=sel_prob,
    audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
)
print(f"Ensemble (2-way, sel/prob) done. {len(ens_sel_prob_records)} segments.")

ens_sel_prob_pooled_records = run_ensemble_inference(
    records, ga_processor, ga_model, en_processor, en_model,
    conf_func=frame_prob_confidence, pool_ga=True,
    selector=sel_prob_pooled,
    audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
)
print(f"Ensemble (2-way, sel/prob+pooled) done. {len(ens_sel_prob_pooled_records)} segments.")

#### 3-way

In [ ]:
ens_3way_sel_gibbs_records        = None
ens_3way_sel_gibbs_pooled_records = None
ens_3way_sel_prob_records         = None
ens_3way_sel_prob_pooled_records  = None

if ru_processor is not None:
    # ── Gibbs confidence ──────────────────────────────────────────────────────
    ens_3way_sel_gibbs_records = run_ensemble_inference(
        records, ga_processor, ga_model, en_processor, en_model,
        ru_processor=ru_processor, ru_model=ru_model,
        conf_func=frame_gibbs_confidence,
        selector=sel_gibbs,
        audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
    )
    print(f"Ensemble (3-way, sel/gibbs) done. {len(ens_3way_sel_gibbs_records)} segments.")

    ens_3way_sel_gibbs_pooled_records = run_ensemble_inference(
        records, ga_processor, ga_model, en_processor, en_model,
        ru_processor=ru_processor, ru_model=ru_model,
        conf_func=frame_gibbs_confidence, pool_ga=True,
        selector=sel_gibbs_pooled,
        audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
    )
    print(f"Ensemble (3-way, sel/gibbs+pooled) done. {len(ens_3way_sel_gibbs_pooled_records)} segments.")

    # ── Probability confidence ────────────────────────────────────────────────
    ens_3way_sel_prob_records = run_ensemble_inference(
        records, ga_processor, ga_model, en_processor, en_model,
        ru_processor=ru_processor, ru_model=ru_model,
        conf_func=frame_prob_confidence,
        selector=sel_prob,
        audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
    )
    print(f"Ensemble (3-way, sel/prob) done. {len(ens_3way_sel_prob_records)} segments.")

    ens_3way_sel_prob_pooled_records = run_ensemble_inference(
        records, ga_processor, ga_model, en_processor, en_model,
        ru_processor=ru_processor, ru_model=ru_model,
        conf_func=frame_prob_confidence, pool_ga=True,
        selector=sel_prob_pooled,
        audio_dir=SEGMENTS_DIR / "audio", device=DEVICE,
    )
    print(f"Ensemble (3-way, sel/prob+pooled) done. {len(ens_3way_sel_prob_pooled_records)} segments.")
else:
    print("RU_MODEL_ID not set — 3-way selector runs skipped.")

In [ ]:
for word in ensemble_3way_records:
    word_details = word['span_details']
    for span in word_details:
        if 'ru' in span['models'].keys():
            print(span)

{'canonical': 'iː', 'predicted': 'pʲ', 'winner': 'ru', 'confidence': 0.9643, 'frames': (25, 29), 'models': {'ga': {'confidence': 0.5169, 'predicted': 'bʲ', 'peak_probs': tensor([3.4293e-03, 2.5062e-03, 1.6576e-03, 7.0334e-03, 3.4808e-03, 4.4523e-02,
        5.1936e-01, 1.4142e-03, 2.8721e-03, 8.4815e-03, 2.5577e-03, 1.2717e-02,
        1.1367e-03, 4.8791e-03, 1.4596e-03, 1.8600e-03, 1.6437e-02, 6.3494e-03,
        3.0281e-02, 3.3262e-03, 6.9445e-03, 1.5652e-03, 2.2700e-03, 4.7507e-03,
        4.5413e-04, 5.5899e-03, 7.1926e-02, 8.6443e-04, 2.3565e-03, 4.8348e-04,
        2.8333e-03, 2.1095e-03, 1.4868e-03, 1.4803e-02, 1.2756e-03, 8.9358e-04,
        1.4938e-03, 6.0936e-03, 4.9511e-03, 2.2707e-02, 1.9299e-03, 9.1545e-03,
        4.9200e-02, 4.7288e-04, 5.5960e-04, 7.2353e-04, 6.3144e-04, 4.1566e-04,
        7.6187e-04, 5.5444e-04, 4.8546e-04, 5.4766e-03, 3.3701e-04, 4.9866e-03,
        1.4551e-03, 1.1744e-03, 4.1926e-04, 8.6740e-04, 3.7294e-04, 4.8673e-04,
        2.7956e-03, 2.2600e-03

In [ ]:
ensemble_3way_records[0]['span_details']

[{'canonical': 'i',
  'predicted': 'ʔ',
  'winner': 'en',
  'confidence': 0.914,
  'frames': (0, 7),
  'models': {'ga': {'confidence': 0.5526,
    'predicted': 'eː',
    'peak_probs': tensor([2.0671e-03, 9.6148e-03, 8.1856e-03, 1.0830e-03, 6.4669e-03, 3.6659e-04,
            1.4285e-03, 1.9427e-02, 5.3886e-04, 1.4530e-03, 9.9591e-02, 2.8786e-01,
            2.6545e-04, 1.2833e-03, 1.4299e-03, 2.4308e-03, 2.8026e-01, 4.5785e-03,
            4.9040e-02, 2.0888e-03, 2.1066e-03, 1.3006e-03, 4.0790e-04, 3.0661e-03,
            7.2577e-05, 3.1309e-04, 1.2716e-03, 2.2463e-04, 1.2407e-03, 9.7116e-05,
            6.2484e-03, 2.0640e-03, 4.3024e-04, 3.2281e-03, 3.8951e-04, 9.4520e-04,
            3.4149e-03, 1.3091e-03, 4.8243e-04, 2.5515e-03, 1.9614e-04, 2.3579e-04,
            8.4154e-04, 7.0773e-05, 1.2494e-04, 7.8475e-05, 1.3060e-04, 7.5523e-05,
            1.4111e-03, 7.9089e-05, 6.9227e-05, 1.6985e-02, 8.3632e-05, 4.2095e-03,
            6.0524e-04, 1.8084e-04, 7.7354e-05, 3.2425e-04, 7.08

## 3.2 Synthetic-augmentation

### 3.2.1 Synthetic-augmented model (adversarial sampling)

In [ ]:
l2_records = None

if l2_model is not None:
    l2_records = []
    for r in records:
        data, sr = sf.read(r['audio_path'], dtype="float32", always_2d=False)
        if sr != 16000:
            data = torchaudio.functional.resample(torch.from_numpy(data).unsqueeze(0), sr, 16000).squeeze(0).numpy()
        result = dict(r)
        result["asr"] = forced_ctc_phones(l2_processor, l2_model, data, r['canonical'], device=DEVICE)
        l2_records.append(result)
    print(f"Done. {len(l2_records)} segments inferred.")
else:
    print("L2_MODEL_ID not set — skipping. Set it in the config cell when the model is ready.")

Done. 148 segments inferred.


### 3.2.2 Synthetic-augmented model (unpaired baseline)

Same architecture as 3.2.1 but trained on synthetic and canonical data as independent instances,
without adversarial canonical/L2 pairing. Serves as an ablation to isolate the contribution
of the pairing strategy.

In [ ]:
l2_baseline_records = None

if l2_baseline_model is not None:
    l2_baseline_records = []
    for r in records:
        data, sr = sf.read(r['audio_path'], dtype="float32", always_2d=False)
        if sr != 16000:
            data = torchaudio.functional.resample(torch.from_numpy(data).unsqueeze(0), sr, 16000).squeeze(0).numpy()
        result = dict(r)
        result["asr"] = forced_ctc_phones(l2_baseline_processor, l2_baseline_model, data, r['canonical'], device=DEVICE)
        l2_baseline_records.append(result)
    print(f"Done. {len(l2_baseline_records)} segments inferred.")
else:
    print("L2_BASELINE_MODEL_ID not set — skipping.")

Done. 148 segments inferred.


## 3.3 Baselines

### 3.3.1 Irish-only (no augmentation)

The un-augmented baseline: plain CTC greedy decode from the Irish model alone,
no ensemble, no synthetic data. This is the starting point the L2-gen approach
builds on — everything else should beat this or the experiment hasn't worked.

In [ ]:
irish_only_records = []
for r in records:
    data, sr = sf.read(r['audio_path'], dtype="float32", always_2d=False)
    if sr != 16000:
        data = torchaudio.functional.resample(torch.from_numpy(data).unsqueeze(0), sr, 16000).squeeze(0).numpy()
    result = dict(r)
    result["asr"] = forced_ctc_phones(ga_processor, ga_model, data, r['canonical'], device=DEVICE)
    irish_only_records.append(result)
print(f"Done. {len(irish_only_records)} segments inferred.")

Done. 148 segments inferred.


### 3.3.2 Irish+English combined (single model)

Single model trained on the union of Irish and English phoneme data, without
ensemble architecture. Tests whether diverse training data alone accounts for
any improvement, or whether the per-span confidence gating of the ensemble is
what matters. Requires a separately trained model — set `L2_COMBINED_MODEL_ID`
when ready.

In [ ]:
l2_combined_records = None

if combined_model is not None:
    l2_combined_records = []
    for r in records:
        data, sr = sf.read(r['audio_path'], dtype="float32", always_2d=False)
        if sr != 16000:
            data = torchaudio.functional.resample(torch.from_numpy(data).unsqueeze(0), sr, 16000).squeeze(0).numpy()
        result = dict(r)
        result["asr"] = forced_ctc_phones(combined_processor, combined_model, data, r['canonical'], device=DEVICE)
        l2_combined_records.append(result)
    print(f"Done. {len(l2_combined_records)} segments inferred.")
else:
    print("COMBINED_MODEL_ID not set — skipping.")

Done. 148 segments inferred.


# 4 Per item Eval

In [ ]:
# Central registry — all 16 factorial configurations + synthesis systems.
# Any system whose records are None is automatically skipped below.
SYSTEMS = {
    # ── Synthesis experiment ──────────────────────────────────────────────────
    "Irish only (baseline)":        irish_only_records,
    "L2 model (unpaired)":          l2_baseline_records,
    "L2 model (paired)":            l2_records,
    # ── Ensemble experiment — no selector ─────────────────────────────────────
    "Irish+English combined":               l2_combined_records,
    "Ensemble (2-way, gibbs)":              ensemble_2way_records,
    "Ensemble (2-way, gibbs+pooled)":       ensemble_2way_pooled_records,
    "Ensemble (2-way, prob)":               ensemble_2way_prob_records,
    "Ensemble (2-way, prob+pooled)":        ensemble_2way_prob_pooled_records,
    "Ensemble (3-way, gibbs)":              ensemble_3way_records,
    "Ensemble (3-way, gibbs+pooled)":       ensemble_3way_pooled_records,
    "Ensemble (3-way, prob)":               ensemble_3way_prob_records,
    "Ensemble (3-way, prob+pooled)":        ensemble_3way_prob_pooled_records,
    # ── Ensemble experiment — with selector ───────────────────────────────────
    "Ensemble (2-way, sel/prob)":           ens_sel_prob_records,
    "Ensemble (2-way, sel/prob+pooled)":    ens_sel_prob_pooled_records,
    "Ensemble (2-way, sel/gibbs)":          ens_sel_gibbs_records,
    "Ensemble (2-way, sel/gibbs+pooled)":   ens_sel_gibbs_pooled_records,
    "Ensemble (3-way, sel/prob)":           ens_3way_sel_prob_records,
    "Ensemble (3-way, sel/prob+pooled)":    ens_3way_sel_prob_pooled_records,
    "Ensemble (3-way, sel/gibbs)":          ens_3way_sel_gibbs_records,
    "Ensemble (3-way, sel/gibbs+pooled)":   ens_3way_sel_gibbs_pooled_records,
}

active_systems = {name: recs for name, recs in SYSTEMS.items() if recs is not None}
print("Active systems:", list(active_systems.keys()))

Active systems: ['Irish only (baseline)', 'L2 model (unpaired)', 'L2 model (paired)', 'Irish+English combined', 'Ensemble (2-way, gibbs)', 'Ensemble (2-way, gibbs+pooled)', 'Ensemble (2-way, prob)', 'Ensemble (2-way, prob+pooled)', 'Ensemble (3-way, gibbs)', 'Ensemble (3-way, gibbs+pooled)', 'Ensemble (3-way, prob)', 'Ensemble (3-way, prob+pooled)', 'Ensemble (2-way, sel/prob)', 'Ensemble (2-way, sel/prob+pooled)', 'Ensemble (2-way, sel/gibbs)', 'Ensemble (2-way, sel/gibbs+pooled)', 'Ensemble (3-way, sel/prob)', 'Ensemble (3-way, sel/prob+pooled)', 'Ensemble (3-way, sel/gibbs)', 'Ensemble (3-way, sel/gibbs+pooled)']


## 4.1 Save Inference Results

Run the **save** cell once after completing all inference above.
On subsequent runs, skip sections 1.3–3.3 (model loading and inference) and run the **load** cell 4.2 below instead — `active_systems` will be restored from disk with all tensors intact.

In [ ]:
import pickle

RESULTS_PATH = PROJECT_ROOT / "data" / "eval" / "inference_results.pkl"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_PATH, "wb") as f:
    pickle.dump(active_systems, f)
print(f"Saved {len(active_systems)} systems to {RESULTS_PATH}")

Saved 20 systems to /home/peter/Desktop/thesis/ThesisProject/data/eval/inference_results.pkl
